this file will clean the dataset and define the `severity_index` output variable
that will be used as ground truth for evaluating our fuzzy logic system

In [ ]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv("../data/globalterrorismdb_0718dist.csv", encoding="latin-1", low_memory=False)

cols = [
    "iyear", "region_txt", "country_txt",
    "attacktype1_txt", "weaptype1_txt",
    "nkill", "nwound", "propextent", "success"
]

df = df_raw[cols].copy()
print(f"Raw rows: {len(df):,}")

In [ ]:
# fill missing casualties with 0
df["nkill"] = df["nkill"].fillna(0)
df["nwound"] = df["nwound"].fillna(0)

# propextent: 4 = unknown, fill missing with 4
df["propextent"] = df["propextent"].fillna(4)

# drop unknown attack types
df = df[df["attacktype1_txt"] != "Unknown"].reset_index(drop=True)

print(f"Clean rows: {len(df):,}")
print(df.isnull().sum())

In [ ]:
# encode attacktype1_txt to numeric scale 1-4 based on lethality
attack_encoding = {
    "Assassination":                        2,
    "Hijacking":                            2,
    "Hostage Taking (Barricade Incident)":  2,
    "Hostage Taking (Kidnapping)":          2,
    "Facility/Infrastructure Attack":       1,
    "Unarmed Assault":                      1,
    "Armed Assault":                        3,
    "Bombing/Explosion":                    4,
}

# encode weaptype1_txt to numeric scale 1-4 based on lethality
weapon_encoding = {
    "Melee":              1,
    "Sabotage Equipment": 1,
    "Unknown":            1,
    "Vehicle (not to include vehicle-borne explosives, i.e., car or truck bombs)": 1,
    "Incendiary":         2,
    "Firearms":           3,
    "Explosives":         4,
    "Chemical":           4,
}

df["attack_encoded"] = df["attacktype1_txt"].map(attack_encoding).fillna(1)
df["weapon_encoded"] = df["weaptype1_txt"].map(weapon_encoding).fillna(1)

print("Attack type encoding:")
print(df["attack_encoded"].value_counts())
print()
print("Weapon type encoding:")
print(df["weapon_encoded"].value_counts())

## defining severity index

we define `severity_index` as a composite score from five factors:

| Component | Weight | Reasoning |
|---|---|---|
| Fatalities (`nkill`) | 0.35 | Most critical indicator |
| Injuries (`nwound`) | 0.25 | Secondary human impact |
| Property damage (`propextent`) | 0.15 | Physical/economic impact |
| Attack type (`attack_encoded`) | 0.15 | Method of attack |
| Weapon type (`weapon_encoded`) | 0.10 | Type of weapon used |

`propextent` in GTD is coded as:
- 1 = Catastrophic (>$1B)
- 2 = Major
- 3 = Minor
- 4 = Unknown

we invert it (4 -> 0, 1 -> 3) so higher = more damage

Final score is normalized to 0-100 range, then binned into:
- **Low** (0-25)
- **Medium** (25-50)
- **High** (50-75)
- **Critical** (75-100)

In [ ]:
# normalize nkill and nwound using 95th percentile as cap
kill_cap  = df["nkill"].quantile(0.95)
wound_cap = df["nwound"].quantile(0.95)

nkill_norm  = (df["nkill"].clip(upper=kill_cap) / kill_cap) * 100
nwound_norm = (df["nwound"].clip(upper=wound_cap) / wound_cap) * 100

# invert propextent: 1->3, 2->2, 3->1, 4->0
prop_inverted = df["propextent"].map({1: 3, 2: 2, 3: 1, 4: 0})
prop_norm = (prop_inverted / 3) * 100

# normalize attack and weapon encoded (range 1-4)
attack_norm = ((df["attack_encoded"] - 1) / 3) * 100
weapon_norm = ((df["weapon_encoded"] - 1) / 3) * 100

# weighted composite score back to casualty-focused weights
df["severity_score"] = (
    0.50 * nkill_norm  +
    0.30 * nwound_norm +
    0.10 * prop_norm   +
    0.05 * attack_norm +
    0.05 * weapon_norm
)

# bin into severity categories
def categorize(score):
    if score < 25:
        return "Low"
    elif score < 50:
        return "Medium"
    elif score < 75:
        return "High"
    else:
        return "Critical"

df["severity_index"] = df["severity_score"].apply(categorize)
print(df["severity_index"].value_counts())
print()
print(df[["nkill", "nwound", "propextent", "attack_encoded", "weapon_encoded",
          "severity_score", "severity_index"]].head(10))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

order = ["Low", "Medium", "High", "Critical"]
sns.countplot(data=df, x="severity_index", order=order, palette="Reds", ax=axes[0])
axes[0].set_title("Severity Index Distribution")
axes[0].set_xlabel("Category")

df["severity_score"].hist(bins=50, ax=axes[1], color="#c0392b", edgecolor="white")
axes[1].set_title("Severity Score Distribution (0-100)")
axes[1].set_xlabel("Score")

plt.tight_layout()
plt.show()

In [ ]:
df.to_csv("../data/gtd_processed.csv", index=False)
print("Saved to data/gtd_processed.csv")
print(f"Final shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

## Summary

- Raw dataset: **181,691 rows**, 135 columns
- After removing unknown attack types: **174,415 rows**
- No missing values remain after cleaning
- Added 2 new encoded input variables: `attack_encoded` and `weapon_encoded`
- Output variable `severity_index` now defined from 5 weighted components:
  - 35% fatalities (`nkill`)
  - 25% injuries (`nwound`)
  - 15% property damage (`propextent`, inverted)
  - 15% attack type (`attack_encoded`)
  - 10% weapon type (`weapon_encoded`)
- Processed file saved locally to `data/gtd_processed.csv` (not pushed to GitHub)
- Ready for fuzzy membership function design